# MgO on Fe. Boron. 

In [2]:
import os
import numpy as np

from agox import AGOX
from agox.environments import Environment
from agox.generators import RattleGenerator
from agox.databases import Database
from agox.models.descriptors.fingerprint import Fingerprint
from agox.models.GPR import GPR
from agox.models.GPR.kernels import RBF, Noise, Constant as C
from agox.models.GPR.priors import Repulsive
from agox.samplers import KMeansSampler
from agox.collectors import ParallelCollector, StandardCollector
from agox.acquisitors import LowerConfidenceBoundAcquisitor
from agox.postprocessors import ParallelRelaxPostprocess, RelaxPostprocess
from agox.helpers import SubprocessGPAW
from agox.evaluators import LocalOptimizationEvaluator
from agox.samplers import FixedSampler

from ase import Atoms
from ase.constraints import FixAtoms
from ase.build import surface, bulk
from ase.io import read, write

from scripts.build_mgo_stack import build_mgo_stack
from scripts.build_fe_stack import build_fe_stack
from scripts.hetero_struct_randomize import HeteroStructRandomize
from scripts.plot_structure import plot_structure
from scripts.build_heteroStruct import build_heteroStruct
from scripts.remove_random_atoms_by_species import remove_random_atoms_by_species
from scripts.add_adsorbate_to_hollows import add_adsorbate_to_hollows

vacuum = 20
a_mgo = 4.212
a_fe = 2.870190   # Fe optimized lcao lattice constant (Å)

a_mgo_matched = a_mgo / np.sqrt(2)
strain = (a_mgo_matched - a_fe) / a_fe * 100

"""
Control Strain
0.0 = Fe lattice
1.0 = Fe stretch to fit MgO

Simul: 
0, 0.25, 0.5, 0.75, 1
"""
interpolation_factor = 0
a_custom = a_fe + interpolation_factor * (a_mgo_matched - a_fe)

dist_z_fe2o = 0.5 # experimental: 2.3 A
ncores = 24 #24; 16 cores for genkai
supercell = (5 , 5, 1)
kpts = (1, 1, 1)

kappa=2
N_iterations = 100

mgo_layer_number = 1
fe_layer_number = 1
confinement_cell_height_multiplyer = 4 # multiply env cell height

"""
Concenstration controls. 
Removing atoms.
5x5 1 monolayer is 25 atoms (Fe layer). Remove 25 remove one monolayer.
"""
removed_num = 0 

num_candidates={0:[20,0], 10:[10,10], 25:[0,20]}
sample_size = 20

for seed in range(103):
	print(F"Start seed: {seed}")
	
	path_result = f"seed_{seed}/0_result"
	path_xsf = f"{path_result}/0_xsf"
	path_fig = f"{path_result}/1_fig"
	db_dir = f"seed_{seed}/1_db"
	latt_log = f'{path_result}/latt_log.md'

	for d in [path_xsf, path_fig, db_dir]:
		os.makedirs(d, exist_ok=True)
		
	with open(latt_log, 'w') as f:
		f.write(f"{a_fe=}\n{a_mgo=}\n{a_mgo_matched=}\n{strain=:.2f}%\n")
		
	bulk_mgo = bulk('MgO', 'rocksalt', a=a_mgo, cubic=True)
	slab_mgo = surface(bulk_mgo, (0,0,1), layers=1, vacuum=vacuum)

	"""
	Calculating distance between MgO layer (Top Mg and bottom O).
	"""
	z_positions = slab_mgo.get_positions()[:, 2]
	unique_z = np.unique(np.round(z_positions, 5))
	
	if len(unique_z) >= 2:
		unique_z.sort()
		dist_mgo = unique_z[1] - unique_z[0]
	
	fe_bulk = bulk('Fe', 'bcc', a=a_custom, cubic=True)
	slab_fe_base = surface(fe_bulk, (0, 0, 1), layers=1, vacuum=vacuum)
	
	slab_mgofe = build_mgo_stack(slab_fe_base, num_layers=mgo_layer_number, dist_mgo=dist_mgo, vacuum=vacuum, output_path=f"{path_xsf}/slab_mgofe.xsf")
	slab_mgo = slab_mgofe[[atom.symbol != 'Fe' for atom in slab_mgofe]].repeat(supercell)
	slab_fe = build_fe_stack(slab_fe_base, num_layers=fe_layer_number, vacuum=vacuum, output_path=f"{path_xsf}/slab_fe.xsf").repeat(supercell)
	
	slab_deposition = slab_fe.copy() 
	slab_substrate = slab_mgo.copy()
    
	# Consentrations controls
	slab_deposition = remove_random_atoms_by_species(slab_deposition, 'Fe', removed_num)

	z_coordinates = slab_deposition.positions[:, 2]
	bottom_height = np.min(z_coordinates)

	# Boron controls
	slab_deposition = add_adsorbate_to_hollows(slab_deposition, symbol='B',height=0, num_atoms=5,seed=42)

	# Build test
	test_heteroStruct, test_substrate, test_deposition, substrate_layer_heights, deposition_layer_heights = build_heteroStruct(slab_substrate, slab_deposition, output_path=f'{path_xsf}/heteroStruct.xsf')

	slab_substrate.pbc = [True, True, False]
	confinement_corner = np.array([0, 0, slab_substrate.positions[:, 2].max() + dist_z_fe2o])
	
	z_pos = slab_deposition.get_positions()[:, 2]
	h_dep = max(z_pos.max() - z_pos.min(), 2.1)
	confinement_cell = slab_deposition.cell.copy()
	confinement_cell[2, 2] = h_dep * confinement_cell_height_multiplyer

	environment = Environment(
		template=slab_substrate,
		symbols=slab_deposition.get_chemical_formula(),
		confinement_cell=confinement_cell,
		confinement_corner=confinement_corner,
		box_constraint_pbc=[True, True, False]
	)
	
	n_rattle = len(slab_deposition)
	generators = [
		HeteroStructRandomize(
			**environment.get_confinement(),
			slab_deposition=slab_deposition,
			hetero_slab_dist=dist_z_fe2o,
			rattle_amplitude=1.5,
			n_rattle=n_rattle,
			generate_pristine=False,
			write_struct=True,
		),
		RattleGenerator(
			**environment.get_confinement(),
			n_rattle=int(n_rattle * 0.5),
			rattle_amplitude=2.3
		),
	]
	
	hetero_candidate = generators[0](sampler=None, environment=environment)[0]
	write(f'{path_xsf}/hetero_candidate.xsf', hetero_candidate)
	
	sampler = FixedSampler(hetero_candidate)
	rattle_candidate = generators[1](sampler, environment)[0]
	write(f'{path_xsf}/rattle_candidate.xsf', rattle_candidate)
	
	database = Database(filename=f"{db_dir}/db_{seed}.db", order=5)
	descriptor = Fingerprint(environment=environment)
	
	beta = 0.01
	kernel = C(5000, (1, 1e5)) * (C(beta, (beta, beta)) * RBF() + C(1-beta, (1-beta, 1-beta)) * RBF()) + Noise(0.01, (0.01, 0.01))
	model = GPR(descriptor=descriptor, kernel=kernel, database=database, prior=Repulsive())
	
	sampler = KMeansSampler(descriptor=descriptor, database=database, sample_size=sample_size)
	collector = ParallelCollector(
		generators=generators,
		sampler=sampler,
		environment=environment,
		num_candidates=num_candidates,
		order=1
	)
	
	acquisitor = LowerConfidenceBoundAcquisitor(model=model, kappa=kappa, order=3)
	
	relaxer = ParallelRelaxPostprocess(
		model=acquisitor.get_acquisition_calculator(),
		constraints=environment.get_constraints(),
		optimizer_run_kwargs={"steps": 100},
		start_relax=10,
		order=2
	)
	
	calc = SubprocessGPAW(
		ncores=ncores,
		mode={"name": "lcao"},
		basis="dzp",
		xc="PBE",
		mixer={"backend": "pulay", "beta": 0.05, "nmaxold": 5, "weight": 100},
		convergence={"energy": 1e-4, "density": 1e-3, "eigenstates": 1e-3},
		txt=f"output_seed_{seed}.txt",
		kpts=kpts,
		symmetry='off',
		nbands='nao',
		maxiter=100,
		occupations={"name": "fermi-dirac", "width": 0.05},
		hund=True,
		spinpol=True
	)
	
	evaluator = LocalOptimizationEvaluator(
		calc,
		gets={"get_key": "prioritized_candidates"},
		optimizer_run_kwargs={"fmax": 0.05, "steps": 1},
		constraints=environment.get_constraints(),
		store_trajectory=False,
		order=4
	)
	
	agox = AGOX(collector, relaxer, acquisitor, evaluator, database, seed=seed)
	agox.run(N_iterations=N_iterations)

Start seed: 0


╭───────────────────────────── Environment report ─────────────────────────────╮
│ Atoms in search:                                                             │
│     B = 5                                                                    │
│     Fe = 25                                                                  │
│ Template formula: Mg25O25                                                    │
│ Full formula: B5Fe25Mg25O25                                                  │
│ Cell:                                                                        │
│     14.35 0.00 0.00                                                          │
│     0.00 14.35 0.00                                                          │
│     0.00 0.00 20.00                                                          │
│ Periodicity:                                                                 │
│     True True False                                                          │
│ Box constraint: True                                                         │
│ Confinement corner                                                           │
│     0.00 0.00 10.50                                                          │
│ Confinement cell:                                                            │
│     14.35 0.00 0.00                                                          │
│     0.00 14.35 0.00                                                          │
│     0.00 0.00 8.40                                                           │
╰──────────────────────────────────────────────────────────────────────────────╯

[HeteroStructRandomize] Saving generated structures to: generated_structures/run_20260413_121343
GPR: Attaching to database: <agox.databases.database.Database object at 0x769b044c2810>
SamplerKMeans: Attaching to database: <agox.databases.database.Database object at 0x769b044c2810>


Numpy random seed: 0

Modules in pool

...

Number of candidates this iteration: 10

───────────────────────────────── PoolRelaxer ──────────────────────────────────

(Actor pid=108345) Traceback (most recent call last):
(Actor pid=108345)   File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/agox/postprocessors/ray_relax/remote_relax.py", line 83, in remote_relax
(Actor pid=108345)     initial_energy = _candidate.get_potential_energy()
(Actor pid=108345)                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(Actor pid=108345)   File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ase/atoms.py", line 835, in get_potential_energy
(Actor pid=108345)     energy = self._calc.get_potential_energy(self)
(Actor pid=108345)              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(Actor pid=108345)   File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ase/calculators/abc.py", line 26, in get_potential_energy
(Actor pid=108345)     return self.get_property(name, atoms)
(Actor pid=108345)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(Actor pid=108345)   File "/home/zeus/miniconda3/envs/cloudspace/lib/pytho

RayTaskError(AttributeError): [36mray::Actor.execute_function()[39m (pid=108347, ip=10.192.11.225, actor_id=f343a3c70194b3c6232ce3f001000000, repr=<agox.utils.ray.actor.Actor object at 0x770ca3b84230>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/agox/utils/ray/actor.py", line 43, in execute_function
    return fn(*[self.modules[key] for key in module_keys], *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/agox/postprocessors/ray_relax/remote_relax.py", line 99, in remote_relax
    forces = _candidate.get_forces()
             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ase/atoms.py", line 886, in get_forces
    forces = self._calc.get_forces(self)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

2026-04-13 12:19:59,964	ERROR worker.py:430 -- Unhandled error (suppress with 'RAY_IGNORE_UNHANDLED_ERRORS=1'): ray::Actor.execute_function() (pid=108346, ip=10.192.11.225, actor_id=08cb5e9a40cb2868591baae501000000, repr=<agox.utils.ray.actor.Actor object at 0x7d7938450ce0>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/agox/utils/ray/actor.py", line 43, in execute_function
    return fn(*[self.modules[key] for key in module_keys], *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/agox/postprocessors/ray_relax/remote_relax.py", line 99, in remote_relax
    forces = _candidate.get_forces()
             ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ase/atoms.py", line 886, in get_forces
    forces = se

# Amorph

In [ ]:
import os
import numpy as np

from ase import Atoms
from ase.build import bulk
from ase.io import write

from agox import AGOX
from agox.environments import Environment
from agox.generators import RattleGenerator
from agox.samplers import KMeansSampler, FixedSampler
from agox.collectors import ParallelCollector, StandardCollector
from agox.acquisitors import LowerConfidenceBoundAcquisitor
from agox.postprocessors import ParallelRelaxPostprocess
from agox.evaluators import LocalOptimizationEvaluator
from agox.databases import Database
from agox.models.descriptors.fingerprint import Fingerprint
from agox.models.GPR import GPR
from agox.models.GPR.kernels import RBF, Noise, Constant as C
from agox.models.GPR.priors import Repulsive
from agox.helpers import SubprocessGPAW

from scripts.amorph_struct_randomize import AmorphStructRandomize
from scripts.amorph_permutation import AmorphPermutationGenerator
from scripts.add_B_concentration import add_B_concentration

a_ta = 3.30

"""
Cell volume controls
"""
scale_cell = 1.00 # 1.05->+5%

supercell = (3, 3, 3)
kpts = (1, 1, 1) 
ncores = 1 # 16 genkai
cB = 0.01 # B concentrations (%)
N_iterations = 100 # AGOX iterations
	
path_result = "0_simul"
path_xsf = f"{path_result}/0_xsf"
path_fig = f"{path_result}/1_fig"
db_dir = f"{path_result}/1_db"

for p in [path_result, path_xsf, path_fig, db_dir]:
	os.makedirs(p, exist_ok=True)

ta_bulk = bulk("Ta", "bcc", a=a_ta, cubic=True) * supercell
ta_bulk.set_cell(ta_bulk.cell * scale_cell, scale_atoms=True)

tab_bulk, n_b, n_ta = add_B_concentration(
	ta_bulk, cB=cB, make_structure=True
)

"""
Environment is an empty cells. Size of ta_bulk. 
"""
template = Atoms("", cell=ta_bulk.cell.copy(), pbc=True)
environment = Environment(
	template=template,
	symbols=tab_bulk.get_chemical_formula(),
	confinement_cell=template.cell.copy(),
	confinement_corner=np.array([0, 0, 0]),
	box_constraint_pbc=[True, True, True],
)

# num_candidates = {0: [10, 0, 0], 10: [5, 5, 0], 25: [0, 5, 5]}
num_candidates = {0: [10]}
sample_size = 10

n_rattle = len(ta_bulk)
generators = [
	AmorphStructRandomize(
		**environment.get_confinement(),
		amorph=tab_bulk,
		write_struct=True,
		rattle_amplitude=1.5,
		n_rattle=n_rattle,
		generate_pristine=False,
	
	),
	AmorphPermutationGenerator(
		**environment.get_confinement(),
		max_number_of_swaps=1,
		rattle_strength=0.0,
	),
	RattleGenerator(
		**environment.get_confinement(),
		n_rattle=int(n_rattle), # 0.7 * n_rattle
		rattle_amplitude=2.3,
	)
]

"""

for i in range(100):
	amorph_candidate = generators[0](sampler=None, environment=environment)[0]
	write(f"{path_xsf}/amorph_candidate_{i}.xsf", amorph_candidate)
	print(f"amorph {i}")
	
	sampler = FixedSampler(amorph_candidate)
	write(f"{path_xsf}/permut_candidate_{i}.xsf", generators[1](sampler, environment)[0])
	print(f"permut {i}")
	
	write(f"{path_xsf}/rattle_candidate_{i}.xsf", generators[2](sampler, environment)[0])
	print(f"rattle {i}")

"""

amorph_candidate = generators[0](sampler=None, environment=environment)[0]
write(f"{path_xsf}/amorph_candidate.xsf", amorph_candidate)

sampler = FixedSampler(amorph_candidate)
write(f"{path_xsf}/permut_candidate.xsf", generators[1](sampler, environment)[0])
write(f"{path_xsf}/rattle_candidate.xsf", generators[2](sampler, environment)[0])

database = Database(filename=f"{db_dir}/db_0.db", order=5)
descriptor = Fingerprint(environment=environment)

beta = 0.01
k0 = C(beta, (beta, beta)) * RBF()
k1 = C(1 - beta, (1 - beta, 1 - beta)) * RBF()
kernel = C(5000, (1, 1e5)) * (k0 + k1) + Noise(0.01, (0.01, 0.01))

model = GPR(
	descriptor=descriptor,
	kernel=kernel,
	database=database,
	prior=Repulsive(),
	# use_ray = False
)

sampler = KMeansSampler(
	descriptor=descriptor,
	database=database,
	sample_size=sample_size,
)

collector = ParallelCollector(
	generators=generators,
	sampler=sampler,
	environment=environment,
	num_candidates=num_candidates,
	order=1,
)

"""
collector = StandardCollector(
	generators=generators,
	sampler=sampler,
	environment=environment,
	num_candidates=num_candidates,
	order=1,
)	
"""

acquisitor = LowerConfidenceBoundAcquisitor(model=model, kappa=2, order=3)

relaxer = ParallelRelaxPostprocess(
	model=acquisitor.get_acquisition_calculator(),
	constraints=environment.get_constraints(),
	optimizer_run_kwargs={"steps": 100},
	start_relax=10,
	order=2,
)

calc = SubprocessGPAW(
	ncores=ncores,
	mode={"name": "lcao"},
	basis="dzp",
	xc="PBE",
	kpts=kpts,
	symmetry="off",
	nbands="nao",
	mixer={"backend": "pulay", "beta": 0.05, "nmaxold": 5, "weight": 100},
	convergence={"energy": 1e-4, "density": 1e-3, "eigenstates": 1e-3},
	occupations={"name": "fermi-dirac", "width": 0.05},
	maxiter=100,
	txt="output.txt",
)

evaluator = LocalOptimizationEvaluator(
	calc,
	gets={"get_key": "prioritized_candidates"},
	optimizer_kwargs={"logfile": None},
	optimizer_run_kwargs={"fmax": 0.05, "steps": 1},
	constraints=environment.get_constraints(),
	order=4,
)

agox = AGOX(
	collector,
	relaxer,
	acquisitor,
	evaluator,
	database,
	seed=1,
)

agox.run(N_iterations=N_iterations)

# MgO on Fe. 5x5. Lattice constraint. 

In [ ]:

import os
import numpy as np

from agox import AGOX
from agox.environments import Environment
from agox.generators import RattleGenerator
from agox.databases import Database
from agox.models.descriptors.fingerprint import Fingerprint
from agox.models.GPR import GPR
from agox.models.GPR.kernels import RBF, Noise, Constant as C
from agox.models.GPR.priors import Repulsive
from agox.samplers import KMeansSampler
from agox.collectors import ParallelCollector, StandardCollector
from agox.acquisitors import LowerConfidenceBoundAcquisitor
from agox.postprocessors import ParallelRelaxPostprocess, RelaxPostprocess
from agox.helpers import SubprocessGPAW
from agox.evaluators import LocalOptimizationEvaluator
from agox.samplers import FixedSampler

from ase import Atoms
from ase.constraints import FixAtoms
from ase.build import surface, bulk
from ase.io import read, write

from scripts.build_mgo_stack import build_mgo_stack
from scripts.build_fe_stack import build_fe_stack
from scripts.hetero_struct_randomize import HeteroStructRandomize
from scripts.plot_structure import plot_structure
from scripts.build_heteroStruct import build_heteroStruct
from scripts.remove_random_atoms_by_species import remove_random_atoms_by_species

# from icecream import ic

vacuum = 20
a_mgo = 4.212
a_fe = 2.866

a_mgo_matched = a_mgo / np.sqrt(2)
strain = (a_mgo_matched - a_fe) / a_fe * 100

"""
Control Strain
0.0 = Fe lattice
1.0 = Fe stretch to fit MgO

Simul: 
0, 0.25, 0.5, 0.75, 1
"""
interpolation_factor = 0
a_custom = a_fe + interpolation_factor * (a_mgo_matched - a_fe)

dist_z_fe2o = 0.5 # experimental: 2.3 A
ncores = 24 #24; 16 cores for genkai
supercell = (5 , 5, 1)
kpts = (1, 1, 1)

kappa=2
N_iterations = 100

mgo_layer_number = 1
fe_layer_number = 1
confinement_cell_height_multiplyer = 4 # multiply env cell height

"""
Concenstration controls. 
Removing atoms.
5x5 1 monolayer is 25 atoms (Fe layer). Remove 25 remove one monolayer.
"""
removed_num = 0 

num_candidates={0:[20,0], 10:[10,10], 25:[0,20]}
sample_size = 20

for seed in range(103):
	print(F"Start seed: {seed}")
	
	path_result = f"seed_{seed}/0_result"
	path_xsf = f"{path_result}/0_xsf"
	path_fig = f"{path_result}/1_fig"
	db_dir = f"seed_{seed}/1_db"
	latt_log = f'{path_result}/latt_log.md'
	
	for d in [path_xsf, path_fig, db_dir]:
		os.makedirs(d, exist_ok=True)
		
	with open(latt_log, 'w') as f:
		f.write(f"{a_fe=}\n{a_mgo=}\n{a_mgo_matched=}\n{strain=:.2f}%\n")
		
	bulk_mgo = bulk('MgO', 'rocksalt', a=a_mgo, cubic=True)
	slab_mgo = surface(bulk_mgo, (0,0,1), layers=1, vacuum=vacuum)
	
	"""
	Calculating distance between MgO layer (Top Mg and bottom O).
	"""
	z_positions = slab_mgo.get_positions()[:, 2]
	unique_z = np.unique(np.round(z_positions, 5))
	
	if len(unique_z) >= 2:
		unique_z.sort()
		dist_mgo = unique_z[1] - unique_z[0]
	
	"""
	Building MgO/Fe(001). Started by making the Fe base (constraint lattice), then put O and Mg on top of it. 
	
	Make MgO(001). We can remove the Fe base.
	Make Fe(001). We multiply the Fe base.
	"""	
	fe_bulk = bulk('Fe', 'bcc', a=a_custom, cubic=True)
	slab_fe_base = surface(fe_bulk, (0, 0, 1), layers=1, vacuum=vacuum)
	
	slab_mgofe = build_mgo_stack(slab_fe_base, num_layers=mgo_layer_number, dist_mgo=dist_mgo, vacuum=vacuum, output_path=f"{path_xsf}/slab_mgofe.xsf")
	slab_mgo = slab_mgofe[[atom.symbol != 'Fe' for atom in slab_mgofe]].repeat(supercell)
	slab_fe = build_fe_stack(slab_fe_base, num_layers=fe_layer_number, vacuum=vacuum, output_path=f"{path_xsf}/slab_fe.xsf").repeat(supercell)
	
	slab_deposition = slab_mgo.copy()
	slab_substrate = slab_fe.copy()
	
	# Consentrations controls
	slab_deposition = remove_random_atoms_by_species(slab_deposition, 'Fe', removed_num)
	
	build_heteroStruct(slab_substrate, slab_deposition, output_path=f'{path_xsf}/heteroStruct.xsf')
	
	slab_substrate.pbc = [True, True, False]
	confinement_corner = np.array([0, 0, slab_substrate.positions[:, 2].max() + dist_z_fe2o])
	
	z_pos = slab_deposition.get_positions()[:, 2]
	h_dep = max(z_pos.max() - z_pos.min(), 2.1)
	confinement_cell = slab_deposition.cell.copy()
	confinement_cell[2, 2] = h_dep * confinement_cell_height_multiplyer
	
	environment = Environment(
		template=slab_substrate,
		symbols=slab_deposition.get_chemical_formula(),
		confinement_cell=confinement_cell,
		confinement_corner=confinement_corner,
		box_constraint_pbc=[True, True, False]
	)
	
	n_rattle = len(slab_deposition)
	generators = [
		HeteroStructRandomize(
			**environment.get_confinement(),
			slab_deposition=slab_deposition,
			hetero_slab_dist=dist_z_fe2o,
			rattle_amplitude=1.5,
			n_rattle=n_rattle,
			generate_pristine=False,
			write_struct=True,
		),
		RattleGenerator(
			**environment.get_confinement(),
			n_rattle=int(n_rattle * 0.5),
			rattle_amplitude=2.3
		),
	]
	
	hetero_candidate = generators[0](sampler=None, environment=environment)[0]
	write(f'{path_xsf}/hetero_candidate.xsf', hetero_candidate)
	
	sampler = FixedSampler(hetero_candidate)
	rattle_candidate = generators[1](sampler, environment)[0]
	write(f'{path_xsf}/rattle_candidate.xsf', rattle_candidate)
	
	database = Database(filename=f"{db_dir}/db_{seed}.db", order=5)
	descriptor = Fingerprint(environment=environment)
	
	beta = 0.01
	kernel = C(5000, (1, 1e5)) * (C(beta, (beta, beta)) * RBF() + C(1-beta, (1-beta, 1-beta)) * RBF()) + Noise(0.01, (0.01, 0.01))
	model = GPR(descriptor=descriptor, kernel=kernel, database=database, prior=Repulsive())
	
	sampler = KMeansSampler(descriptor=descriptor, database=database, sample_size=sample_size)
	collector = ParallelCollector(
		generators=generators,
		sampler=sampler,
		environment=environment,
		num_candidates=num_candidates,
		order=1
	)
	
	acquisitor = LowerConfidenceBoundAcquisitor(model=model, kappa=kappa, order=3)
	
	relaxer = ParallelRelaxPostprocess(
		model=acquisitor.get_acquisition_calculator(),
		constraints=environment.get_constraints(),
		optimizer_run_kwargs={"steps": 100},
		start_relax=10,
		order=2
	)
	
	calc = SubprocessGPAW(
		ncores=ncores,
		mode={"name": "lcao"},
		basis="dzp",
		xc="PBE",
		mixer={"backend": "pulay", "beta": 0.05, "nmaxold": 5, "weight": 100},
		convergence={"energy": 1e-4, "density": 1e-3, "eigenstates": 1e-3},
		txt=f"output_seed_{seed}.txt",
		kpts=kpts,
		symmetry='off',
		nbands='nao',
		maxiter=100,
		occupations={"name": "fermi-dirac", "width": 0.05},
		hund=True,
		spinpol=True
	)
	
	evaluator = LocalOptimizationEvaluator(
		calc,
		gets={"get_key": "prioritized_candidates"},
		optimizer_run_kwargs={"fmax": 0.05, "steps": 1},
		constraints=environment.get_constraints(),
		store_trajectory=False,
		order=4
	)
	
	agox = AGOX(collector, relaxer, acquisitor, evaluator, database, seed=seed)
	agox.run(N_iterations=N_iterations)

# Dos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from gpaw import GPAW, FermiDirac
from ase.dft.dos import DOS
from ase.io import read

def calculate_and_compare_dos(structures, kpts=(4, 4, 1), seeds=None, savename="does_results.csv"):
    if seeds is None:
        seeds = [i for i in range(len(structures))]
    
    # Format: { 'energy_seed_101': [...], 'dos_seed_101': [...], ... }
    all_data = {}

    for atoms, seed in zip(structures, seeds):
        print(f"run seed: {seed}")
    
        calc = GPAW(
            mode={"name": "lcao"},
            basis="dzp",
            xc="PBE",
            mixer={"backend": "pulay", "beta": 0.05, "nmaxold": 5, "weight": 100},
            convergence={"energy": 1e-4, "density": 1e-3, "eigenstates": 1e-3},
            txt=f"output_seed_{seed}.txt",
            kpts=kpts,
            symmetry='off',
            nbands='nao',
            maxiter=300,
            occupations={"name": "fermi-dirac", "width": 0.05},
            hund=True,
            spinpol=True
        )

        atoms.calc = calc
        atoms.get_potential_energy()
        e_fermi = calc.get_fermi_level()

        dos_obj = DOS(calc, npts=800, width=0.1)
        energies = dos_obj.get_energies() - e_fermi  

        dos_up = np.zeros_like(energies)
        dos_down = np.zeros_like(energies)
        total_dos = np.zeros_like(energies)

        if calc.get_number_of_spins() == 2:
            dos_up = dos_obj.get_dos(spin=0)
            dos_down = dos_obj.get_dos(spin=1)
            total_dos = dos_up + dos_down
        else:
            total_dos = dos_obj.get_dos()
            dos_up = total_dos
        
        results_storage[f'energy_{seed}'] = energies
        results_storage[f'total_dos_{seed}'] = total_dos
        results_storage[f'spin_up_{seed}'] = dos_up
        results_storage[f'spin_down_{seed}'] = dos_down

    
    df = pd.DataFrame(results_storage)
    df.to_csv(savename, index=False)
    print(f"saved: {savename}")

    return results_storage
    
structs = read(f"traj_19.traj", index=':')
results = calculate_and_compare_dos(structs, kpts=(4, 4, 1))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_dos_from_csv(csv_path="dos_data.csv", x_range=[-10, 5]):

    df = pd.read_csv(csv_path)

    seeds = [col.replace('energy_', '') for col in df.columns if col.startswith('energy_')]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

    for seed in seeds:
        E = df[f'energy_{seed}']
        total_dos = df[f'total_dos_{seed}']
        spin_up = df[f'spin_up_{seed}']
        spin_down = df[f'spin_down_{seed}']

        ax1.plot(E, total_dos, label=f"Total Seed {seed}", lw=1.8)
        ax2.plot(E, spin_up, label=f"Seed {seed} $\uparrow$", lw=1.5)
        ax2.plot(E, -spin_down, label=f"Seed {seed} $\downarrow$", lw=1.5, linestyle='--')
    
    ax1.set_title("Total Density of States", fontsize=14, fontweight='bold')
    ax1.set_ylabel("DOS (states/eV)", fontsize=12)
    ax1.axvline(0, color='black', linestyle=':', alpha=0.8, lw=1)
    ax1.legend(loc='upper right', frameon=False, ncol=2)
    ax1.grid(visible=True, linestyle=':', alpha=0.5)

    ax2.set_title("Spin-Polarized DOS ($\uparrow$ vs $-\downarrow$)", fontsize=14, fontweight='bold')
    ax2.set_ylabel("DOS (states/eV)", fontsize=12)
    ax2.set_xlabel("Energy - $E_F$ (eV)", fontsize=12)
    ax2.axvline(0, color='black', linestyle=':', alpha=0.8, lw=1)

    ax2.axhline(0, color='black', lw=0.8) # Baseline for spin split
    ax2.legend(loc='upper right', frameon=False, ncol=2)
    ax2.grid(visible=True, linestyle=':', alpha=0.5)

    plt.xlim(x_range)
    plt.tight_layout()
    # plt.savefig("dos_comparison_plot.png", dpi=300)
    plt.show()

plot_dos_from_csv("dos_data.csv", x_range=[-8, 4])

